In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install auto-round --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os
import shutil

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor



In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "Cloudflare/clef"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 25 files: 100%|██████████████████████| 25/25 [00:00<00:00, 561.14it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
         

In [9]:
TUNING_CONFIG = {
    "group_size": 64,
    "sym": True,
    "iters": 800,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        # Exempt linear convolution kernels from quantization
        "*.linear_conv": {"data_type": "bfloat16"},
        "*.conv1d": {"data_type": "bfloat16"}
    }
}

In [10]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-10-03 16:22:19 WARNING autoround.py L489: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-03 16:22:19 WARNING autoround.py L489: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-03 16:22:19 WARNING autoround.py L489: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [11]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-10-03 16:22:20 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30,32-34,36-38,40-42,44-46,48-50,52-54,56-58,60-62].linear_attn.in_proj_a, model.language_model.layers.[0-2,4-6,8-10,12-14,16-18,20-22,24-26,28-30,32-34,36-38,40-42,44-46,48-50,52-54,56-58,60-62].linear_attn.in_proj_b, model.visual.blocks.[0-26].mlp.linear_fc1, model.visual.blocks.[0-26].mlp.linear_fc2
2026-10-03 16:22:20 WARNING logging.py L340: Layer name or regex '*.linear_conv' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-10-03 16:22:20 WARNING logging.py L340: Layer name or regex '*.conv1d' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-10-03 16:22:20 INFO base.py L1365: `torch.compile` is enabled
2026-10-03 16:22:21 INFO orchestrator.py L594:

Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.linear_conv': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.conv1d': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.linear_conv': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.conv1d': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.linear_conv': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.conv1d': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.linear_conv': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING common.py L955: Skipping invalid regex pattern '*.conv1d': nothing to repeat at position 0
2026-10-03 17:28:24 WARNING 

Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

2026-10-03 17:32:08 INFO device.py L1560: 'peak_ram': 69.21GB, 'peak_vram': 51.09GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1152)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-26): 27 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1152, out_features=3456, bias=True)
             (proj): Linear(in_features=1152, out_features=1152, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
             (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
             (act_fn): 

In [16]:
# Files that must accompany the quantized backbone for downstream inference
auxiliary_files = [
    "joint_head.safetensors",
    "joint_head_config.json",
    "joint_schema_model.py",
    "chat_template.jinja"
]

export_subdirs = [
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-round-auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/llm-compressor-wint-a16"),
]

for export_dir in export_subdirs:
    if os.path.exists(export_dir):
        print(f"\n--> Injecting Clef joint head assets into: {export_dir}")
        for filename in auxiliary_files:
            src = os.path.join(LOCAL_PATH, filename)
            dst = os.path.join(export_dir, filename)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"    Copied: {filename}")
            else:
                print(f"    Warning: Missing {src}")


--> Injecting Clef joint head assets into: ./AutoRound/local_model-w4g64/auto-round-auto-gptq
    Copied: joint_head.safetensors
    Copied: joint_head_config.json
    Copied: joint_schema_model.py
    Copied: chat_template.jinja

--> Injecting Clef joint head assets into: ./AutoRound/local_model-w4g64/auto-gptq
    Copied: joint_head.safetensors
    Copied: joint_head_config.json
    Copied: joint_schema_model.py
    Copied: chat_template.jinja

--> Injecting Clef joint head assets into: ./AutoRound/local_model-w4g64/llm-compressor-wint-a16
    Copied: joint_head.safetensors
    Copied: joint_head_config.json
    Copied: joint_schema_model.py
    Copied: chat_template.jinja


In [17]:
def push_to_hub(local_dir, repo_name, token):
    if not os.path.exists(local_dir):
        print(f"[Hub] Directory not found, skipping: {local_dir}")
        return

    full_repo_id = f"{HF_USER}/{repo_name}"
    try:
        api = HfApi()
        print(f"[Hub] Preparing repo: {full_repo_id}...")
        create_repo(full_repo_id, repo_type="model", private=False, token=token, exist_ok=True)
        
        print(f"[Hub] Uploading folder from {local_dir}...")
        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            token=token
        )
        print(f"[Hub] Upload complete: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] Error uploading {full_repo_id}: {e}")


In [18]:

base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()


In [20]:

if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/auto-gptq"), 
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g64/llm-compressor-wint-a16"), 
        f"{base_name}-W4A16-AutoRound-LLM-Compressor", 
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload.")

[Hub] Preparing repo: Vishva007/clef-W4A16-AutoRound...


[Hub] Uploading folder from ./AutoRound/local_model-w4g64/auto-round-auto-gptq...
[Hub] Upload complete: https://huggingface.co/Vishva007/clef-W4A16-AutoRound
[Hub] Preparing repo: Vishva007/clef-W4A16-AutoRound-GPTQ...
[Hub] Uploading folder from ./AutoRound/local_model-w4g64/auto-gptq...
[Hub] Upload complete: https://huggingface.co/Vishva007/clef-W4A16-AutoRound-GPTQ
[Hub] Preparing repo: Vishva007/clef-W4A16-AutoRound-LLM-Compressor...
[Hub] Uploading folder from ./AutoRound/local_model-w4g64/llm-compressor-wint-a16...
[Hub] Upload complete: https://huggingface.co/Vishva007/clef-W4A16-AutoRound-LLM-Compressor
